In [1]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import yfinance as yf
import talib

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
gc = yf.Ticker("GC=F")
data = gc.history('max')

In [3]:
# Calculate the split point
split_idx = int(len(data) * 0.8)

# Split the data
iss_data = data.iloc[:split_idx]
oss_data = data.iloc[split_idx:]

print(f"ISS Range: {iss_data.index[0]} to {iss_data.index[-1]}")
print(f"OSS Range: {oss_data.index[0]} to {oss_data.index[-1]}")

ISS Range: 2000-08-30 00:00:00-04:00 to 2021-03-12 00:00:00-05:00
OSS Range: 2021-03-15 00:00:00-04:00 to 2026-04-27 00:00:00-04:00


In [60]:
class dema_buy(Strategy):

    fast_window = 10
    slow_window = 30
    filter_window = 65

    def init(self):
        self.fast = self.I(talib.DEMA, self.data.Close, self.fast_window)
        self.slow = self.I(talib.DEMA, self.data.Close, self.slow_window)
        self.filter = self.I(talib.DEMA, self.data.Close, self.filter_window)

    def next(self):
        price = self.data.Close[-1]

        if crossover(self.fast, self.slow) and price>self.filter:
            self.buy()

        elif crossover(self.slow, self.fast) or price<self.filter:
            self.position.close()

In [61]:
bt = Backtest(iss_data, dema_buy, cash=10_000)

In [66]:
stats = bt.optimize(
    fast_window = range(7,14,1),
    slow_window = range(26,34,1),
    max_tries = 100,
    maximize = "Equity Final [$]"
)

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/56 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

... (49 outputs left out)

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5020 [00:00<?, ?bar/s]

In [67]:
stats

Start                     2000-08-30 00:00...
End                       2021-03-12 00:00...
Duration                   7499 days 01:00:00
Exposure Time [%]                    26.06331
Equity Final [$]                  21531.70093
Equity Peak [$]                    23329.1012
Return [%]                          115.31701
Buy & Hold Return [%]               557.80415
Return (Ann.) [%]                     3.82486
Volatility (Ann.) [%]                 8.70448
CAGR [%]                              2.61075
Sharpe Ratio                          0.43941
Sortino Ratio                         0.65483
Calmar Ratio                          0.22321
Alpha [%]                           -12.35053
Beta                                  0.22888
Max. Drawdown [%]                   -17.13598
Avg. Drawdown [%]                    -2.76068
Max. Drawdown Duration     1992 days 00:00:00
Avg. Drawdown Duration      111 days 00:00:00
# Trades                                   89
Win Rate [%]                      

In [68]:
bt.plot()

GridPlot(id='p3534', ...)